# Predictive Modeling for NYC Airbnb Prices
This notebook trains and evaluates multiple regression models to predict Airbnb listing prices and extracts key pricing features.

In [1]:
import sys
import os
sys.path.append(os.path.abspath('../'))
import pandas as pd
import numpy as np
from src.modeling import prepare_features, train_all_models, evaluate_predictions, plot_residuals, plot_feature_importance

## 1. Load and Preprocess Data
We load the cleaned data, split into features and target, and apply encoding and scaling.

In [2]:
CLEAN_DATA_PATH = '../data/cleaned/AB_NYC_2019_cleaned.csv'
df = pd.read_csv(CLEAN_DATA_PATH)
print(f'Loaded cleaned data shape: {df.shape}')

preprocessor, X_train_proc, X_test_proc, y_train, y_test, feature_names = prepare_features(df)
print(f'Training shape: {X_train_proc.shape}')
print(f'Testing shape: {X_test_proc.shape}')

Loaded cleaned data shape: (48884, 20)
Training shape: (39107, 17)
Testing shape: (9777, 17)


## 2. Train Regression Models
We train Linear Regression, Random Forest, and Gradient Boosting Regressors.

In [3]:
models = train_all_models(X_train_proc, y_train)

Training Linear Regression...
Successfully trained Linear Regression
Training Gradient Boosting...
Successfully trained Gradient Boosting
Training Random Forest...
Successfully trained Random Forest


## 3. Model Evaluation
We evaluate model predictions in actual dollars (reversing the log transformation) for real business interpretability.

In [4]:
results = {}
for name, model in models.items():
    metrics = evaluate_predictions(model, X_test_proc, y_test)
    results[name] = metrics
    print(f'=== {name} ===')
    print(f'Log RMSE: {metrics["log_rmse"]:.4f}')
    print(f'Log R2: {metrics["log_r2"]:.4f}')
    print(f'Dollar RMSE: ${metrics["dollar_rmse"]:.2f}')
    print(f'Dollar MAE: ${metrics["dollar_mae"]:.2f}')
    print(f'Dollar R2: {metrics["dollar_r2"]:.4f}')
    print()

=== Linear Regression ===
Log RMSE: 0.4969
Log R2: 0.4859
Dollar RMSE: $188.20
Dollar MAE: $61.49
Dollar R2: 0.1153

=== Gradient Boosting ===
Log RMSE: 0.4413
Log R2: 0.5945
Dollar RMSE: $179.42
Dollar MAE: $56.17
Dollar R2: 0.1960

=== Random Forest ===
Log RMSE: 0.4453
Log R2: 0.5871
Dollar RMSE: $177.34
Dollar MAE: $56.34
Dollar R2: 0.2145



## 4. Diagnostics and Feature Importance
We plot residuals and feature importances for our best performing model.

In [5]:
# Find the best model based on Dollar MAE
best_model_name = min(results, key=lambda k: results[k]['dollar_mae'])
best_model = models[best_model_name]
best_metrics = results[best_model_name]
print(f'Best Model: {best_model_name}')

FIG_DIR = '../reports/figures'
plot_residuals(best_metrics['actuals_dollar'].values, best_metrics['predictions_dollar'], FIG_DIR)
plot_feature_importance(best_model, feature_names, FIG_DIR)

Best Model: Gradient Boosting
Saved residual plots to ../reports/figures\residuals_and_predictions.png
Saved feature importance plot to ../reports/figures\feature_importance.png


## 5. Model-Driven Business Recommendations
- **Best Model**: The Gradient Boosting Regressor outperforms Linear Regression and Random Forest. It captures non-linear relationships and interactions between listing attributes.
- **Pricing Recommendations for Hosts**:
  1. **Room Type**: Renting out an 'Entire home/apt' adds a massive premium. Hosts should maximize availability for entire properties.
  2. **Location**: Latitude and longitude are highly influential. Listings closer to Manhattan center command higher pricing. Hosts in outer boroughs should lower baseline price to remain competitive.
  3. **Reviews and Engagement**: Listings with more reviews and activity can support higher pricing as they build trust. Encourage guests to review to gain pricing power.